# Catchment stats and forest coverage

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"

In [ ]:
land_use = gpd.read_file(base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp")
print(land_use.crs)

In [ ]:
catchments_unionized_final = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
print("Catchments:", len(catchments))


In [ ]:
# Basic stats (printed)
sizes = catchments["area_km2"]
print(f"Count: {sizes.size}")
print(f"Min (km²):   {sizes.min():.2f}")
print(f"Mean (km²):  {sizes.mean():.2f}")
print(f"Max (km²):   {sizes.max():.2f}")
print(f"Range (km²): {sizes.max() - sizes.min():.2f}")

# Minimal table to carry forward
out = catchments[["catchment_uid", "area_m2", "area_km2"]].copy()

# Show top 30 smallest by area (km²) with more precision
print("\nTop 30 smallest catchments (km², 8 d.p.):")
print(
    catchments[["catchment_uid","area_km2"]]
      .sort_values("area_km2")
      .head(30)
      .assign(area_km2=lambda d: d["area_km2"].map(lambda x: f"{x:.8f}"))
      .to_string(index=False)
)


# FOREST CATEGORIES

In [ ]:
# === Forest-by-catchment stats (minimal) =====================================
# Constants (names match your usage)
CAT_EXIST, CAT_REFO, CAT_TREAT, CAT_OTHER = "existing_forest","reforestable","treated_as_forest","other"
category_cols = [CAT_EXIST, CAT_REFO, CAT_TREAT, CAT_OTHER]

# Safety: make sure size-stats cell has run
assert "out" in locals() and {"catchment_uid","area_m2","area_km2"}.issubset(out.columns), \
    "Run your size-stats cell first to create `out` with area_m2/area_km2."

# Pick the land-use label column: prefer "Classify" if it exists and matches; else auto
known = (set(existing_forest_classes) | set(reforestable_classes) |
         set(treated_as_forest_classes) | set(mixed_land_use_fractions.keys()))
if "Classify" in land_use.columns and land_use["Classify"].isin(known).any():
    LU_COL = "Classify"
else:
    str_cols = [c for c in land_use.columns if land_use[c].dtype == object]
    LU_COL = max(str_cols, key=lambda c: land_use[c].isin(known).sum())
print("LU_COL:", LU_COL)

# Build class → {category: fraction} mapping (compact)
class_to_frac = {}
for s, group in [(CAT_EXIST, existing_forest_classes),
                 (CAT_REFO,  reforestable_classes),
                 (CAT_TREAT, treated_as_forest_classes)]:
    for lbl in group: class_to_frac.setdefault(lbl, {})[s] = 1.0
for lbl, parts in mixed_land_use_fractions.items():
    for k, v in parts.items():
        if "existing" in k:   class_to_frac.setdefault(lbl, {})[CAT_EXIST] = float(v)
        elif "reforest" in k: class_to_frac.setdefault(lbl, {})[CAT_REFO]  = float(v)
        elif "treated"  in k: class_to_frac.setdefault(lbl, {})[CAT_TREAT] = float(v)
# Any present-but-unmapped class → other:1.0
for lbl in land_use[LU_COL].astype(str).unique():
    if lbl not in class_to_frac or not class_to_frac[lbl]:
        class_to_frac[lbl] = {CAT_OTHER: 1.0}

In [ ]:
# Overlay
inter = gpd.overlay(
    land_use[[LU_COL, "geometry"]],
    catchments[["catchment_uid", "geometry"]],
    how="intersection"
).rename(columns={LU_COL: "lu_class"})
inter["area_m2"] = inter.geometry.area

In [ ]:
# === Allocate mixed classes, aggregate, and write CSV (post-overlay) =========
# Progress: allocation
print("Allocating areas to categories …", flush=True)
rows = []
get = class_to_frac.get
for cls, uid, a in zip(inter["lu_class"].astype(str).values,
                       inter["catchment_uid"].values,
                       inter["area_m2"].values):
    for cat, frac in get(cls, {CAT_OTHER: 1.0}).items():
        if frac:
            rows.append((uid, cat, float(a) * float(frac)))

alloc = pd.DataFrame(rows, columns=["catchment_uid", "category", "area_m2"])
print(f"Allocated rows: {len(alloc):,}", flush=True)
if alloc.empty:
    raise RuntimeError("No allocations produced — check LU_COL and class names.")

# Aggregate → m², km², % of mapped, % of whole catchment
print("Aggregating by catchment/category …", flush=True)
wide_m2 = (
    alloc.groupby(["catchment_uid", "category"], as_index=False)
         .agg(area_m2=("area_m2", "sum"))
         .pivot(index="catchment_uid", columns="category", values="area_m2")
         .reindex(columns=category_cols, fill_value=0.0)
)
print(f"Catchments in table: {len(wide_m2):,} | Categories: {len(wide_m2.columns):,}", flush=True)

cat_m2   = wide_m2.add_suffix("_m2")
total_m2 = cat_m2.sum(axis=1)

cat_km2 = (cat_m2 / 1e6).rename(columns=lambda c: c.replace("_m2", "_km2"))
cat_pct = (cat_m2.div(total_m2.where(total_m2 > 0, pd.NA), axis=0) * 100)\
           .rename(columns=lambda c: c.replace("_m2", "_pct"))

area_m2_series = out.set_index("catchment_uid")["area_m2"]
cat_pct_catch = (cat_m2.div(area_m2_series, axis=0) * 100)\
                 .rename(columns=lambda c: c.replace("_m2", "_of_catchment_pct"))

forest_stats = (
    pd.concat([cat_m2, cat_km2, cat_pct, cat_pct_catch], axis=1)
      .assign(total_m2=total_m2, total_km2=total_m2 / 1e6)
      .reset_index()
      .sort_values("catchment_uid")
)

# Merge into your size table
catchment_table_with_landuse = out.merge(forest_stats, on="catchment_uid", how="left")

# (Optional) 'other' breakdown string — keep if you need it
other_classes = [k for k, v in class_to_frac.items() if v.get(CAT_OTHER, 0) > 0]
if other_classes:
    print("Computing 'other' breakdown …", flush=True)
    other_long = (
        inter[inter["lu_class"].astype(str).isin(other_classes)]
        .groupby(["catchment_uid", "lu_class"], as_index=False)
        .agg(area_m2=("area_m2", "sum"))
    )
    other_long["pct_of_mapped"] = 100 * other_long["area_m2"] / other_long["catchment_uid"].map(total_m2)

    def _summ(g):
        s = g.sort_values("pct_of_mapped", ascending=False)
        s = s[s["pct_of_mapped"] >= 0.05]
        return "; ".join(f"{r.lu_class} ({r.pct_of_mapped:.1f}%)" for _, r in s.head(6).iterrows())

    other_summary = other_long.groupby("catchment_uid").apply(_summ)\
                              .rename("other_breakdown_pct_of_mapped")
    catchment_table_with_landuse = catchment_table_with_landuse.merge(
        other_summary, on="catchment_uid", how="left"
    )

# Write ONCE at the very end
csv_path = output_dir / "catchment_attributes/catchment_landuse_by_category_stats.csv"
catchment_table_with_landuse.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}", flush=True)
print(f"Rows: {len(catchment_table_with_landuse):,} | Cols: {len(catchment_table_with_landuse.columns):,}", flush=True)
print(catchment_table_with_landuse.head(10).to_string(index=False), flush=True)